In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import math
import torch.optim as optim
import itertools
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Global namefiles used for read csv files

In [3]:
global_path_train = 'data/final_train_data_transmission_generalization_lora.csv'
global_path_test  = 'data/final_test_data_transmission_generalization_lora.csv'

# Positional Encoding

In [4]:
class TimePositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # Frecuencias fijas según Attention Is All You Need
        # Fixed frequencies according to Attention is all you need        
        # div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        i = torch.arange(0, d_model//2)
        div_term = torch.exp(-np.log(10000) * (2*i / d_model))
        self.register_buffer('div_term', div_term)

    def forward(self, delta_t):
        # batch x seq x d_model
        pe = torch.zeros(delta_t.size(0), delta_t.size(1), self.d_model, device=delta_t.device)
        # angle = (batch x seq x 1) x ( 1 x 1 x d_model//2)
        angle = delta_t.unsqueeze(-1) * self.div_term.unsqueeze(0).unsqueeze(0)
        pe[:, :, 0::2] = torch.sin(angle)
        pe[:, :, 1::2] = torch.cos(angle)
        return pe

# Model 

In [5]:
class LoraCollisionTransformer(nn.Module):        
    def __init__(self, num_numerical_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        # num_numerical_features- cantidad de variables fisicas (features)    
        # d_model               - Es la dimensión de representación interna del Transformer. Piensa en esto como la cantidad de "canales de información" que el modelo 
        #                         usará internamente para describir cada paquete. Todo el procesamiento profundo se hará en esta dimensión.
        # n_heads               - Le dice al Transformer en cuántas perspectivas diferentes debe dividir su atención.
        # n_layers              - Cuántas veces se repite el proceso de análisis (cuántos bloques Transformer se apilan).
        # dropout               - 
        super().__init__()
        self.num_num_features = num_numerical_features
        self.d_model = d_model
        
        # 1. Proyección directa de las variables físicas a d_model
        # Entran 'num_numerical_features' (ej. 6) y salen 'd_model' (ej. 64)
        # Recibe tu vector original de 6 variables físicas y lo transforma (lo proyecta) matemáticamente en un vector de tamaño d_model (64 dimensiones).
        # Recives the original input vector and project it to a vector size d_model
        self.input_projection = nn.Linear(num_numerical_features, d_model)
        
        # Apply TimePositionalEncoding 
        self.time_encoding = TimePositionalEncoding(d_model)
        
        # Apply encoder_layer
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, 
                                                   dim_feedforward=d_model*4, dropout=dropout, batch_first=True)        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (Batch, Seq_Length, Features = 7) -> 6 físicas + 1 Delta_t
        
        # 1. DESEMPAQUETAR EL TENSOR X
        # Tomamos todas las columnas excepto la última (que es el tiempo)
        x_num = x[:, :, :self.num_num_features] 
        
        # La última columna es el Delta de Tiempo
        delta_t = x[:, :, -1] 
        
        # 2. PROYECCIÓN FÍSICA
        # Convertimos las 6 variables físicas en un vector de 64 dimensiones
        x_projected = self.input_projection(x_num)       
        
        # 3. POSITIONAL ENCODING (Inyección del tiempo)
        time_pe = self.time_encoding(delta_t)
        x_encoded = x_projected + time_pe
        
        # 4. TRANSFORMER
        transformer_out = self.transformer_encoder(x_encoded) 
        
        # 5. CLASIFICACIÓN
        target_packet = transformer_out[:, -1, :]             
        output = self.classifier(target_packet)               
        
        return output.squeeze(-1)

# Functions

In [6]:
def load_and_prepare_data(seq_length=16):    
    print("Loading dataset...")

    df_train = pd.read_csv(global_path_train)
    df_test  = pd.read_csv(global_path_test)

    # Selecting continuous features
    continuous_features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']
    
    X_train_features = df_train[continuous_features].values
    #y_train          = df_train['rcvOk'].values

    X_test_features = df_test[continuous_features].values
    #y_test          = df_test['rcvOk'].values


    scaler = StandardScaler()
    scaler.fit(np.concatenate((X_train_features, X_test_features), axis=0))

    #X_train = scaler.transform(X_train_features)
    #X_test  = scaler.transform(X_test_features)

    def create_sequences_data(df, scaler):
        cols_to_drop = ['dstId', 'srcSat', 'dstSat', 'loraCF', 'loraBW', 'loraCR', 'satId', 'srcId']
        df           = df.drop(columns=cols_to_drop)

        # Sort by id_simulaiton and time
        df = df.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
        
        # Apply StandardScaler         
        df[continuous_features] = scaler.transform(df[continuous_features])

        # Process data
        seq_X   = []
        label_y = []
        for sim_id, group_df in df.groupby('id_simulation'):
            num_features_array  = group_df[continuous_features].values
            ###cat_features_array  = group_df['srcId'].values
            time_array          = group_df['time'].values
            target_array        = group_df['rcvOk'].values
            num_packets         = len(group_df)
            
            # Discard if the simulation has less than SEQ_LENGHTS
            if num_packets < seq_length:
                continue
                        
            # Create sliding windows - Ventanas deslizantes
            for i in range(num_packets - seq_length + 1):
                # Select the window        
                window_num      = num_features_array[i : i + seq_length]
                ###window_cat      = cat_features_array[i : i + SEQ_LENGTH].reshape(-1, 1)
                window_times    = time_array[i : i + seq_length]
                window_targets  = target_array[i : i + seq_length]
                
                # El paquete central a predecir es el último de la ventana
                # The prediction should be the last elemento of the window
                target_idx      = seq_length - 1
                label_target    = window_targets[target_idx]
                time_target     = window_times[target_idx]
                            
                # Si un paquete ocurrió en el mismo segundo que el objetivo, delta_t = 0        
                delta_t = (window_times - time_target).reshape(-1, 1)
                #delta_t =  time_target - window_times
                #print('delta_t: ', delta_t)
                
                #window_X = np.hstack((window_num, window_cat, delta_t))
                window_X = np.hstack((window_num, delta_t))

                seq_X.append(window_X)
                label_y.append(label_target)
                    
        X = torch.tensor(np.array(seq_X), dtype=torch.float32)
        y = torch.tensor(np.array(label_y), dtype=torch.float32)

        return X, y
    

    X_train, y_train = create_sequences_data(df_train, scaler)
    X_test , y_test  = create_sequences_data(df_test, scaler)


    return X_train, y_train, X_test, y_test

In [7]:
def train_test(config, train_loader, test_loader, device, epochs=20, save_logs = False):    
    logs = {}
    logs['train_loss_log']  = []
    logs['test_loss_log']   = []
    logs['train_acc_log']  = []
    logs['test_acc_log']   = []


    # Instanciar modelo con la configuración actual
    model = LoraCollisionTransformer(
        num_numerical_features  = 8, # 8 reales + 1 t_delta (el delta se adiciona despues)
        d_model                 = config['d_model'],
        n_heads                 = config['n_heads'],
        n_layers                = config['n_layers'],
        dropout                 = config['dropout']
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    
    best_test_acc = 0.0    
    for epoch in range(epochs):
        # Train
        model.train()
        train_loss      = 0.0
        correct_train   = 0
        total_train     = 0

        #for X_b, y_b in train_loader:
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Forward pass
            predictions = model(batch_X) # Usa internamente el squeeze(-1)

            # Clean gradient 
            optimizer.zero_grad()

            # Calculate error loss
            loss = criterion(predictions, batch_y)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            # loss.item - The average of the batch losses will give you an estimate of the “epoch loss” during training. Returns the value of this tensor as a standard Python number 
            train_loss += loss.item() * batch_X.size(0)
            
            # Convert to 0 - 1 the vector batch
            predictions_class = (predictions >= 0.5).float()
            
            correct_train += (predictions_class == batch_y).sum().item()
            total_train   += batch_y.size(0)

            
        
        avg_train_loss  = train_loss / total_train
        train_acc       = correct_train / total_train
        if save_logs:
            logs['train_loss_log'].append(avg_train_loss)
            logs['train_acc_log'].append(train_acc)

        # Testing
        model.eval()
        test_loss    = 0.0
        correct_test = 0
        total_test   = 0
        
        y_true = []
        y_pred = []
        with torch.no_grad():
            #for X_b, y_b in val_loader:
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                # Predict
                predictions = model(batch_X)
                loss        = criterion(predictions, batch_y)
                test_loss   += loss.item() * batch_X.size(0)

                # Calculate correct or not 
                predictions_class = (predictions >= 0.5).float()
                correct_test      += (predictions_class == batch_y).sum().item()
                total_test        += batch_y.size(0)

                # Other method to calculate metrics
                y_pred.extend(predictions_class.cpu().numpy())  
                y_true.extend(batch_y.cpu().numpy())
                
        
        avg_test_loss = test_loss / total_test
        test_acc      = correct_test / total_test

        if save_logs:
            logs['test_loss_log'].append(avg_test_loss)
            logs['test_acc_log'].append(test_acc)

        if test_acc > best_test_acc:
            best_test_acc = test_acc
        

        # Print metrics
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} - Acc: {train_acc*100:.1f}% | "
        f"Test Loss: {avg_test_loss:.4f} - Acc: {test_acc*100:.1f}% | " 
        f"Test Acc: {acc:.4f} - Test Prec: {prec*100:.1f}% | "
        f"Test Rec: {rec:.4f} - Test F1: {f1*100:.1f}% | "
        )

    if save_logs:             
        return best_test_acc, logs
    else:
        return best_test_acc

In [8]:
def fine_tuning():
    X_train, y_train, X_test, y_test = load_and_prepare_data(seq_length = 16)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset  = TensorDataset(X_test, y_test)

    device = "cpu"
    print(f"Using : {device}")
    
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = "cpu"
    
    # Setting params
    param_grid = {
        'd_model'   : [32, 64, 128],                # Dimension of the model
        'n_heads'   : [2, 4],                       # Number of projections
        'n_layers'  : [1, 2, 3],                    # Number of encoder layers
        'dropout'   : [0.1, 0.2], # , 0.2                # doprout
        'lr'        : [0.001], #, 0.0005            # Learning rate
        'batch_size': [32]                          # Batch Size
    }
    
    # Generate the combination of parameters
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    print(f"Starting  {len(combinations)} combinations ...")
    print("="*60)
    
    print(combinations)

    results     = []    
    start_total = time.time()    
    for i, config in enumerate(combinations):        
        # Create DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        test_loader  = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
        
        print(f"[{i+1}/{len(combinations)}] Testing: {config} ...", end=" ")
        
        # Entrenar (Usamos pocos epochs para filtrar rápido, ej: 10)
        # Training
        
        #''' 
        final_acc = train_test(config, train_loader, test_loader, device, epochs=20)
        
        print(f"-> Best Accuracy: {final_acc*100:.2f}%")
        
        # Guardar resultado
        res             = config.copy()
        res['accuracy'] = final_acc
        results.append(res)
        #'''
    
    print("="*60)
    print(f"Finishing in {(time.time() - start_total)/60:.1f} minutes.")
        
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by='accuracy', ascending=False)
    
    print("Best results")
    print(results_df.head(5))
    
    # Save to CSV
    results_df.to_csv('hyperparameter_results.csv', index=False)
    
    
    # Return best configuration
    best_config = results_df.iloc[0].to_dict()
    return best_config

In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [10]:
best_params = fine_tuning()
print("Best configuration : ")
print(best_params)

Loading dataset...
Using : cpu
Starting  36 combinations ...
[{'d_model': 32, 'n_heads': 2, 'n_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 2, 'n_layers': 1, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 2, 'n_layers': 2, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 2, 'n_layers': 2, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 2, 'n_layers': 3, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 2, 'n_layers': 3, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 4, 'n_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 4, 'n_layers': 1, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}, {'d_model': 32, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}, {'d_mode

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Epoch 01/20 | Train Loss: 0.2546 - Acc: 89.2% | Test Loss: 0.2160 - Acc: 90.8% | Test Acc: 0.9081 - Test Prec: 75.2% | Test Rec: 0.9199 - Test F1: 82.8% | 
Epoch 02/20 | Train Loss: 0.1111 - Acc: 95.4% | Test Loss: 0.1942 - Acc: 91.5% | Test Acc: 0.9148 - Test Prec: 75.2% | Test Rec: 0.9632 - Test F1: 84.4% | 
Epoch 03/20 | Train Loss: 0.1018 - Acc: 95.9% | Test Loss: 0.1902 - Acc: 91.4% | Test Acc: 0.9143 - Test Prec: 74.8% | Test Rec: 0.9697 - Test F1: 84.4% | 
Epoch 04/20 | Train Loss: 0.0933 - Acc: 96.4% | Test Loss: 0.2263 - Acc: 89.5% | Test Acc: 0.8951 - Test Prec: 69.7% | Test Rec: 0.9957 - Test F1: 82.0% | 
Epoch 05/20 | Train Loss: 0.0932 - Acc: 96.3% | Test Loss: 0.1935 - Acc: 91.7% | Test Acc: 0.9174 - Test Prec: 75.3% | Test Rec: 0.9762 - Test F1: 85.0% | 
Epoch 06/20 | Train Loss: 0.0851 - Acc: 96.5% | Test Loss: 0.1834 - Acc: 92.6% | Test Acc: 0.9258 - Test Prec: 78.5% | Test Rec: 0.9502 - Test F1: 86.0% | 
Epoch 07/20 | Train Loss: 0.0837 - Acc: 96.6% | Test Loss: 0.218

In [ ]:
def simple_execution(config):  
    #X, y = load_and_prepare_data(seq_length = 16)

    X_train, y_train, X_test, y_test = load_and_prepare_data(seq_length = 16)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset  = TensorDataset(X_test, y_test)
    
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = "cpu"
    print(f"Using : {device}")
        
    # Train and test with best params
    train_loader = DataLoader(train_dataset, batch_size=int(config['batch_size']), shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=int(config['batch_size']), shuffle=False)

    final_acc, logs = train_test(config, train_loader, test_loader, device, epochs=20, save_logs=True)
    print(f"Best Test Accuracy: {final_acc*100:.2f}%")

    #train_loss = logs['']
    #test_loss  = logs['']
    print("Final logs: " , logs)
    #print("Final Train loss: " , test_loss)
    return logs


In [ ]:
#type(best_params['d_model'])
best_params

In [ ]:
int_fields = ["d_model", "n_heads", "n_layers", "batch_size"]

for key in int_fields:
    best_params[key] = int(best_params[key])


logs = simple_execution(best_params)

In [ ]:
logs.keys()

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, 21)

plt.figure(figsize=(14, 6))

# ACCURACY
plt.subplot(1, 2, 1)
plt.plot(epochs_range, logs['train_acc_log'], label='Training', marker='o')
plt.plot(epochs_range, logs['test_acc_log'], label='Testing', marker='o', linestyle='--')
plt.title('Accuracy evolution over epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# LOSS
plt.subplot(1, 2, 2)
plt.plot(epochs_range, logs['train_loss_log'], label='Training', marker='o')
plt.plot(epochs_range, logs['test_loss_log'], label='Testing', marker='o', linestyle='--')
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Binary Cross Entropy Loss')
plt.legend(loc='upper right')
plt.grid(True)


plt.suptitle('Acc and Loss for the best config')
plt.tight_layout()
plt.show()

In [ ]:
!conda install matplotlib

In [ ]:
y